In [ ]:
# 必要なものを用意
## ライブラリ
!pip install scikit-learn
## データセット
from datasets import load_dataset
clinc = load_dataset("clinc_oos", "plus")
## 評価関数
from sklearn.metrics import accuracy_score
## パイプライン
from transformers import pipeline
bert_ckpt = "transformersbook/bert-base-uncased-finetuned-clinc"
pipe = pipeline("text-classification", model=bert_ckpt)
## 意図
intents = clinc["test"].features["intent"]
## ベンチマーク用クラス
import torch
from pathlib import Path
from time import perf_counter
import numpy as np
class PerformanceBenchmark:
    def __init__(self, pipeline, dataset, optim_type="BERT baseline"):
        self.pipeline = pipeline
        self.dataset = dataset
        self.optim_type = optim_type

    def compute_accuracy(self):
        preds, labels = [], []
        for example in self.dataset:
            pred = self.pipeline(example["text"])[0]["label"]
            label = example["intent"]
            preds.append(intents.str2int(pred))
            labels.append(label)
        accuracy = accuracy_score(labels, preds)
        print(f"Accuracy on test set - {accuracy:.3f}")
        return {"accuracy": accuracy}

    def compute_size(self):
        temp_path = Path("model.pt")
        torch.save(self.pipeline.model.state_dict(), temp_path)
        size_mb = temp_path.stat().st_size / (1024 * 1024)
        print(f"Model size - {size_mb:.2f} MB")
        temp_path.unlink()
        return {"size_mb": size_mb}

    def time_pipeline(self, query="What is the pin number for my account?"):
        latencies = []
        for _ in range(10):
            _ = self.pipeline(query)
        for _ in range(100):
            start_time = perf_counter()
            _ = self.pipeline(query)
            latency = perf_counter() - start_time
            latencies.append(latency)
        time_avg_ms = 1000 * np.mean(latencies)
        time_std_ms = 1000 * np.std(latencies)
        print(f"Average latency (ms) - {time_avg_ms:.2f} +\- {time_std_ms:.2f}")
        return {"time_avg_ms": time_avg_ms, "time_std_ms": time_std_ms}

    def run_benchmark(self):
        metrics = {}
        metrics[self.optim_type] = self.compute_size()
        metrics[self.optim_type].update(self.time_pipeline())
        metrics[self.optim_type].update(self.compute_accuracy())
        return metrics
## ベースライン
pb = PerformanceBenchmark(pipe, clinc["test"])
perf_metrics = pb.run_benchmark()
## 生徒モデル
finetuned_ckpt = "kirapika2/distilbert-base-uncased-finetuned-clinc"
pipe = pipeline("text-classification", model=finetuned_ckpt)
optim_type = "DistilBERT"
pb = PerformanceBenchmark(pipe, clinc["test"], optim_type=optim_type)
perf_metrics.update(pb.run_benchmark())
## 蒸留モデル
distilled_ckpt = "kirapika2/distilbert-base-uncased-distilled-clinc"
pipe = pipeline("text-classification", model=distilled_ckpt)
optim_type = "Distillation"
pb = PerformanceBenchmark(pipe, clinc["test"], optim_type=optim_type)
perf_metrics.update(pb.run_benchmark())
## プロット
import pandas as pd
import matplotlib.pyplot as plt
def plot_metrics(perf_metrics, current_optim_type):
    df = pd.DataFrame.from_dict(perf_metrics, orient='index')

    for idx in df.index:
        df_opt = df.loc[idx]
        # 現在の最適化タイプを中抜きの円で強調
        if idx == current_optim_type:
            plt.scatter(df_opt["time_avg_ms"], df_opt["accuracy"] * 100,
                        s=df_opt["size_mb"], label=idx,
                        facecolors='none', edgecolors='C0', linewidths=2)
        else:
            plt.scatter(df_opt["time_avg_ms"], df_opt["accuracy"] * 100,
                        s=df_opt["size_mb"], label=idx, alpha=0.5)

    legend = plt.legend(bbox_to_anchor=(1,1))
    for handle in getattr(legend, "legend_handles", getattr(legend, "legendHandles", [])):
        if hasattr(handle, "set_sizes"):
            handle.set_sizes([20])

    plt.ylim(80,90)
    # 最も遅いモデルを使い、x軸のレンジを定義
    xlim = int(perf_metrics["BERT baseline"]["time_avg_ms"] + 3)
    plt.xlim(1,xlim)
    plt.ylabel("Accuracy (%)")
    plt.xlabel("Average latency (ms)")
    plt.show()

In [ ]:
## モデルの量子化
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.quantization import quantize_dynamic
model_ckpt = "kirapika2/distilbert-base-uncased-distilled-clinc"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = (AutoModelForSequenceClassification
         .from_pretrained(model_ckpt).to("cpu"))
model_qunantized = quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)

In [ ]:
# 量子化モデルのベンチマーク
pipe = pipeline("text-classification", model=model_qunantized,
                tokenizer=tokenizer, device=-1)
optim_type = "Distillation + Quantization"
pb = PerformanceBenchmark(pipe, clinc["test"], optim_type=optim_type)
perf_metrics.update(pb.run_benchmark())

In [ ]:
# プロット
plot_metrics(perf_metrics, optim_type)